# Обучение Word2Vec по авторам

Этот ноутбук обучает **отдельную word2vec-модель для каждого автора** на подготовленном корпусе художественных текстов XIX века (см. `preprocessing.ipynb`).

## Входные данные
- `data/lemmas_pos_sentences/<author>.txt` — **1 строка = 1 предложение**, токены разделены пробелами.
- каждый токен имеет вид `lemma_POS`, чтобы в большинстве случаев **развести омонимию** и повысить интерпретируемость (как в статье).

## Параметры обучения (как в практической части статьи)
- **архитектура**: skip-gram (`sg=1`)
- **размерность**: 300 (`vector_size=300`)
- **окно**: 5 (`window=5`)
- **отсечение редких слов**: `min_count=10`
- **negative sampling**: `negative=5`
- **число эпох**: `epochs=5`

## Выходные файлы
- **каноническая модель**: `models/w2v_authors/<author>_w2v.model` и `models/w2v_authors/<author>_w2v.kv`
- **повторные обучения (replicates)** для проверки устойчивости (Приложение А в статье):
  - `models/w2v_authors/replicates/<author>_seed<seed>.model`
  - `models/w2v_authors/replicates/<author>_seed<seed>.kv`


In [ ]:
import multiprocessing
from pathlib import Path

from gensim.models import Word2Vec

BASE = Path.cwd()
SENT_DIR = BASE / 'data' / 'lemmas_pos_sentences'
MODELS_DIR = BASE / 'models' / 'w2v_authors'
REPL_DIR = MODELS_DIR / 'replicates'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPL_DIR.mkdir(parents=True, exist_ok=True)
SENT_DIR, MODELS_DIR, REPL_DIR


In [ ]:
TAG_MAP = {
    # author tagset -> UD-like tags (to match rucorpora)
    "S": "NOUN",
    "V": "VERB",
    "A": "ADJ",
    "ADV": "ADV",
    "NUM": "NUM",
    "INTJ": "INTJ",
    "PART": "PART",
    "CONJ": "CCONJ",
    "PR": "ADP",        # preposition
    "APRO": "DET",      # adjectival pronoun
    "SPRO": "PRON",     # substantive pronoun
    "ADVPRO": "PRON",   # pronominal adverb (closest common UD bucket)
    "ANUM": "NUM",      # adjectival numeral
    "COM": "X",         # rare/unclear in this pipeline
}


def normalize_token(tok: str) -> str:
    if "_" not in tok:
        return tok
    lemma, tag = tok.rsplit("_", 1)
    tag_norm = TAG_MAP.get(tag, tag)
    return f"{lemma}_{tag_norm}"


def load_sentences(path: Path) -> list[list[str]]:
    sentences: list[list[str]] = []
    for line in path.read_text(encoding='utf-8').splitlines():
        toks = [normalize_token(t) for t in line.split()]
        if toks:
            sentences.append(toks)
    return sentences


In [ ]:
def train_author(path: Path, *, seed: int, save_canonical: bool) -> dict:
    author = path.stem
    sentences = load_sentences(path)

    workers = max(1, multiprocessing.cpu_count() - 1)
    model = Word2Vec(
        sentences=sentences,
        vector_size=300,
        window=5,
        min_count=10,
        workers=workers,
        sg=1,
        epochs=5,
        negative=5,
        seed=seed,
    )

    rep_bin = REPL_DIR / f'{author}_seed{seed}.model'
    rep_kv = REPL_DIR / f'{author}_seed{seed}.kv'
    model.save(str(rep_bin))
    model.wv.save_word2vec_format(rep_kv, binary=False)

    if save_canonical:
        can_bin = MODELS_DIR / f'{author}_w2v.model'
        can_kv = MODELS_DIR / f'{author}_w2v.kv'
        model.save(str(can_bin))
        model.wv.save_word2vec_format(can_kv, binary=False)

    return {
        'author': author,
        'seed': int(seed),
        'n_sentences': len(sentences),
        'vocab_size': len(model.wv),
        'rep_kv_path': str(rep_kv),
    }


In [ ]:
SEEDS = [42, 43, 44]
CANONICAL_SEED = 42

results = []
for path in sorted(SENT_DIR.glob('*.txt')):
    for seed in SEEDS:
        info = train_author(path, seed=seed, save_canonical=(seed == CANONICAL_SEED))
        print(
            f"{info['author']} seed={info['seed']}: sentences={info['n_sentences']}, "
            f"|V|={info['vocab_size']} -> {info['rep_kv_path']}"
        )
        results.append(info)

results


## Общеязыковая модель Ruscorpora (референс)

Для сопоставления авторских моделей с «усреднённым» языковым пространством, как в статье, используется внешняя общеязыковая модель **Ruscorpora**.

Ниже мы **скачиваем её из интернета** через каталог `gensim-data` (`gensim.downloader`: модель `word2vec-ruscorpora-300`). Затем сохраняем распакованные вектора в `models/w2v_authors/rucorpora.kv`, чтобы дальше читать их локально так же, как и авторские `.kv`, в `w2v_compare.ipynb`.


In [ ]:
import gensim.downloader as api

MODEL_NAME = "word2vec-ruscorpora-300"
dst = MODELS_DIR / "rucorpora.kv"

# Скачивание из интернета (gensim-data) и загрузка KeyedVectors
kv = api.load(MODEL_NAME)

# Сохраняем распакованные вектора рядом с авторскими моделями
kv.save_word2vec_format(dst, binary=False)

print("Saved:", dst)
